# More on Lists: Algorithms and Mutability
**COMP 170** · Linear search · Reversing lists · Deleting while traversing · Mutability

## 1. Linear Search: Find the First Student Who Passed
A teacher stores quiz scores in submission order. Write `first_passing_score(scores: list[int]) -> int` to return the **index** of the first score ≥ 7, or `-1` if none exists.

```python
first_passing_score([3, 5, 8, 6, 9])  # 2
first_passing_score([7, 4, 9])        # 0
first_passing_score([2, 5, 4])        # -1
first_passing_score([])               # -1
```

**Constraints:** Use indexes and a loop. Do not use `index()`, `count()`, `in`, or search helpers. Try a `for` loop without `break`, then a `while` loop without `break` or `while True`.

### Think it through
- What value signals “not found”?
- How can we preserve the **first** matching index?
- How could we stop checking once we have found a match?

**Linear search** examines elements one at a time. It works even when a list is unsorted; in the worst case it checks every element.

### Two approaches
- A `for` loop with `range(len(scores))` visits every index unless we explicitly change control flow.
- A `while` loop can combine “still in bounds” and “not yet found” in its condition.

**Notes:**

Always check that the index is within the list's bounds before accessing an element.

Python evaluates `and` from left to right and stops when the first condition is false.

**See code:** [linear_search_problem.py](https://github.com/Kennaoui/COMP170_Fall26/blob/main/W_11/linear_search_problem.py)

**Example:** For `scores = [3, 8, 4, 9]`, what index should the function return? What should it return if no score passes?

## 2. Reversing the Elements of a List
Given `[10, 20, 30, 40]`, produce the reversed order `[40, 30, 20, 10]`.

There are **two design choices**:
- **Out of place:** build and return a **new list**; leave the input unchanged.
- **In place:** rearrange elements **inside the original list**; do not create another list to hold the result.

We will implement both using indexes and loops, rather than `reverse()`, `reversed()`, or slicing.

### Approach A — Out of place
Start with an empty result list. Traverse the original list **from right to left** and append each element to the result.

You can traverse the original list backward using either negative indices (from -1 to -len(items)) or positive indices (from len(items) - 1 down to 0).
 
**Trace:** `[10, 20, 30, 40]` → `[]` → `[40]` → `[40, 30]` → `[40, 30, 20]` → `[40, 30, 20, 10]`.

**Question:** Which list is changed? What does the function return?

### Approach B — In place
Swap the first and last elements, then the second and second-to-last, and continue **until the middle**.

**Trace:**
```text
[10, 20, 30, 40]
[40, 20, 30, 10]  swap indexes 0 and 3
[40, 30, 20, 10]  swap indexes 1 and 2
```

Python can swap two list elements with temp variable or using tuple-swapping `items[left], items[right] = items[right], items[left]`. The right side forms a tuple of the two values, which is then unpacked into the two positions.

**Question:** Why do we stop at the middle instead of swapping across the entire list?

**See code:** [resversed_list.py](https://github.com/Kennaoui/COMP170_Fall26/blob/main/W_11/resversed_list.py)

### Compare the approaches
| Question | Out of place | In place |
|---|---|---|
| Creates a new result list? | Yes | No |
| Changes the input list? | No | Yes |
| Return value | New list | `None` |

**Try it:** Predict the result for an empty list, a one-element list, and a list with an odd number of elements.

## 3. Problem 3: Remove Failing Scores While Traversing

Write `remove_failing_scores(scores: list[int]) -> None` to remove **every score below 7**, modifying the original list.

```python
scores = [4, 3, 8, 5, 9, 2]
remove_failing_scores(scores)
print(scores)  # [8, 9]
```

**Constraints:** Do not use `filter()` or create a replacement list.

First, consider this tempting solution. What could go wrong?

In [ ]:
def remove_failing_scores_wrong(scores: list[int]) -> None:
    for i in range(len(scores)):
        if scores[i] < 7:
            scores.pop(i)

# Predict what happens before running this:
scores = [4, 3, 8, 5, 9, 2]
remove_failing_scores_wrong(scores)
print("List after the attempt:", scores)


### Why does this fail?
`range(len(scores))` is determined **before** the loop starts. But `pop(i)` shortens the list and shifts later elements one position to the left.

For example, removing `4` at index 0 from `[4, 3, 8, 5, 9, 2]` produces `[3, 8, 5, 9, 2]`. The next loop index is 1, so the `3` now at index 0 is **skipped**. Eventually the loop may try to access an index that no longer exists, causing an `IndexError`.

**Key idea:** After deleting an element, the element that shifts into its position has **not yet been checked**.

### Correct approach — Control the index with `while`
Keep an index `i` pointing to the next score to inspect.
- If `scores[i] < 7`, remove it **without increasing `i`**.
- Otherwise, increase `i` to move forward.
- Check `i < len(scores)` each time because the list may shrink.

**Trace:**
```text
[4, 3, 8, 5, 9, 2]   i=0: remove 4; i stays 0
[3, 8, 5, 9, 2]      i=0: remove 3; i stays 0
[8, 5, 9, 2]         i=0: keep 8; i becomes 1
[8, 5, 9, 2]         i=1: remove 5; i stays 1
[8, 9, 2]            i=1: keep 9; i becomes 2
[8, 9, 2]            i=2: remove 2; i stays 2
[8, 9]               Stop: i == len(scores)
```

In [ ]:
def remove_failing_scores(scores: list[int]) -> None:
    i = 0
    while i < len(scores):
        if scores[i] < 7:
            scores.pop(i)
        else:
            i += 1


**Try it:** What happens for `[2, 4, 6]`, `[7, 8, 9]`, and `[]`? Why must `i` remain unchanged after `pop(i)`?

## 4. Mutability: What Does It Mean?
An object is **mutable** if we can change its contents **after it is created**, without replacing it with a new object. An **immutable** object cannot be changed in place.

In Python, a variable refers to an object. **Changing an object** and **assigning a variable to a different object** are not the same thing.

Types we have encountered:
- **Mutable:** `list`, Objects created from our own classes, ...
- **Immutable:** `int`, `float`, `bool`, `str`, and `tuple`...

For example, `numbers[0] = 99` changes a list, whereas `text[0] = "H"` is not allowed for a string.

In [ ]:
numbers = [1, 2, 3]
numbers[0] = 99        # Allowed: a list is mutable
print(numbers)

word = "cat"
word = "b" + word[1:]  # Allowed: assign a NEW string to word
print(word)
# word[0] = "b"       # TypeError: strings are immutable

### Consequence 1 — Two variables can refer to the same list
Assignment with `=` does **not** automatically copy a list. If two variables refer to the same list, a mutation through either variable is visible through both.

By contrast, `copy = original.copy()` makes a new outer list. (For nested lists, the inner objects can still be shared.)

In [ ]:
original = [1, 2, 3]
alias = original       # Both names refer to the same list
alias.append(4)
print(original)        # [1, 2, 3, 4]

independent = original.copy()
independent.append(5)
print(original)        # [1, 2, 3, 4]
print(independent)     # [1, 2, 3, 4, 5]

### Consequence 2 — Functions can modify the caller’s list
When a list is passed to a function, the function receives a reference to that same list object. Mutating it changes what the caller sees, **even if the function returns `None`**.

However, assigning a new list to the function’s local parameter name does not replace the caller’s variable.

**Question:** Why might it matter whether a function changes its input?

In [ ]:
def add_zero(items: list[int]) -> None:
    items.append(0)

values = [1, 2]
add_zero(values)
print(values)  # [1, 2, 0]


def reassign_locally(items: list[int]) -> None:
    items = [99]  # Only the local name is reassigned

reassign_locally(values)
print(values)  # Still [1, 2, 0]

## 5. Connect Mutability to Our Three List Operations

| Operation | Reads or modifies the original list? | Why? |
|---|---|---|
| **Linear search** | Reads only | It checks values and returns an index; it never changes list elements. |
| **Reversing out of place** | Reads original; creates a new list | Appending changes the **new result list**, not the input. |
| **Reversing in place** | Modifies original | Swapping assigns new values to existing positions. |
| **Removing failing scores while traversing** | Modifies original | `pop(i)` removes a score and shifts later elements. |

**Takeaway:** A list being mutable does **not** mean every operation on it mutates it. The algorithm determines whether we **observe** the list, **build a new one**, or **change the existing one**.

### Final prediction exercise
Before running the code, predict the three printed lists. Which operations affect `data` and why?

In [ ]:
data = [3, 8, 5, 9]
other = data
copy_backwards = data[::-1]  # Creates a new reversed list
remove_failing_scores(other)
print("data:", data)
print("other:", other)
print("copy_backwards:", copy_backwards)

